# TraitAlign Kaggle Master Trainer

Instructions:
1. Ensure GPU is turned on (T4x2 or A100).
2. Add the generated `movielens_traitalign_data.zip` or `lastfm_traitalign_data.zip` as a dataset in the Kaggle UI.
3. Change `DATASET_NAME` and `DATA_DIR` below to match your setup.
4. Run all cells!

In [ ]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ['TORCH_USE_CUDA_DSA'] = "1"

import torch

print("1. Installing PyTorch Geometric...")
!pip install -q torch_geometric optuna tqdm scikit-learn

pt_version = torch.__version__
print(f"2. Detected Kaggle PyTorch Version: {pt_version}")

url = f"https://data.pyg.org/whl/torch-{pt_version}.html"
print(f"Fetching matching PyG binaries from: {url}")

!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f {url}

print("✅ INSTALLATION COMPLETE! No restart required!")

In [ ]:
import sys
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv, GATConv, HeteroConv
from torch_geometric.data import HeteroData
from torch_geometric.loader import LinkNeighborLoader
import torch_geometric.transforms as T
import optuna
import pandas as pd
import numpy as np
import time
from tqdm.auto import tqdm
from sklearn.metrics import ndcg_score

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
from scipy.stats import spearmanr, entropy
from sklearn.preprocessing import minmax_scale
from sklearn.metrics.pairwise import cosine_similarity
import gc
print("✅ INSTALLATION COMPLETE! No restart required!")


## 1. PyTorch Architecture

In [ ]:
class ManualHeteroGNN(nn.Module):
    def __init__(self, hidden_channels, gnn_type="graphsage"):
        super().__init__()
        
        if gnn_type == "graphsage":
            conv1_layer = lambda: SAGEConv((-1, -1), hidden_channels)
            conv2_layer = lambda: SAGEConv((-1, -1), hidden_channels)
        elif gnn_type == "gcn":
            from torch_geometric.nn import GraphConv
            conv1_layer = lambda: GraphConv((-1, -1), hidden_channels)
            conv2_layer = lambda: GraphConv((-1, -1), hidden_channels)
        elif gnn_type == "gat":
            conv1_layer = lambda: GATConv((-1, -1), hidden_channels, heads=4, dropout=0.1, add_self_loops=False)
            conv2_layer = lambda: GATConv((-1, -1), hidden_channels, heads=4, dropout=0.1, concat=False, add_self_loops=False)
        elif gnn_type == "lightgcn":
            from torch_geometric.nn import LGConv
            conv1_layer = lambda: LGConv()
            conv2_layer = lambda: LGConv()
            
        self.conv1 = HeteroConv({
            ('user', 'rates', 'item'): conv1_layer(),
            ('item', 'rev_rates', 'user'): conv1_layer(),
        }, aggr='sum')
        
        self.conv2 = HeteroConv({
            ('user', 'rates', 'item'): conv2_layer(),
            ('item', 'rev_rates', 'user'): conv2_layer(),
        }, aggr='sum')
        
        self.gnn_type = gnn_type
        
    def forward(self, x_dict, edge_index_dict, edge_weight_dict=None):
        x_dict_0 = x_dict
        
        # Pass global edge weights if the layer relies on degree normalization
        if self.gnn_type in ["gcn", "lightgcn"] and edge_weight_dict is not None:
            x_dict_1 = self.conv1(x_dict_0, edge_index_dict, edge_weight_dict=edge_weight_dict)
        else:
            x_dict_1 = self.conv1(x_dict_0, edge_index_dict)
        
        if self.gnn_type == "gat":
            x_dict_1 = {k: F.elu(x) for k, x in x_dict_1.items()}
        elif self.gnn_type != "lightgcn":
            x_dict_1 = {k: F.relu(x) for k, x in x_dict_1.items()}
            
        if self.gnn_type in ["gcn", "lightgcn"] and edge_weight_dict is not None:
            x_dict_2 = self.conv2(x_dict_1, edge_index_dict, edge_weight_dict=edge_weight_dict)
        else:
            x_dict_2 = self.conv2(x_dict_1, edge_index_dict)
        
        if self.gnn_type == "lightgcn":
            out_dict = {}
            for k in x_dict_0.keys():
                out_dict[k] = (x_dict_0[k] + x_dict_1[k] + x_dict_2[k]) / 3.0
            return out_dict
            
        return x_dict_2

class TraitAlignMLP(nn.Module):
    def __init__(self, minilm_dim=384, scalar_dim=2, compressed_dim=16, hidden_dim=64, out_dim=5):
        super().__init__()
        self.minilm_dim = minilm_dim
        if minilm_dim > 0:
            self.minilm_compressor = nn.Linear(minilm_dim, compressed_dim)
            combined_dim = compressed_dim + scalar_dim
        else:
            combined_dim = scalar_dim
        self.net = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, out_dim)
        )
    def forward(self, minilm_vec, scalar_vec):
        if self.minilm_dim > 0:
            compressed_minilm = self.minilm_compressor(minilm_vec)
            combined = torch.cat([compressed_minilm, scalar_vec], dim=1)
        else:
            combined = scalar_vec
        return self.net(combined)

class TraitAlignWrapper(nn.Module):
    def __init__(self, num_users, num_items, gnn_type="graphsage", embed_dim=64, minilm_dim=384, scalar_dim=2, out_dim=5):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        
        self.gnn = ManualHeteroGNN(hidden_channels=embed_dim, gnn_type=gnn_type)
        self.mlp = TraitAlignMLP(minilm_dim=minilm_dim, scalar_dim=scalar_dim, out_dim=out_dim)
        
    def forward(self, x_dict, edge_index_dict, edge_weight_dict=None):
        u_idx = torch.clamp(x_dict['user'], min=0, max=self.user_emb.num_embeddings - 1)
        i_idx = torch.clamp(x_dict['item'], min=0, max=self.item_emb.num_embeddings - 1)
        
        x_dict_emb = {
            'user': self.user_emb(u_idx),
            'item': self.item_emb(i_idx)
        }
        return self.gnn(x_dict_emb, edge_index_dict, edge_weight_dict)
        
    def bpr_loss(self, user_feat, pos_item_feat, neg_item_feat, users, pos_items, neg_items):
        pos_scores = (user_feat * pos_item_feat).sum(dim=1)
        neg_scores = (user_feat * neg_item_feat).sum(dim=1)
        bpr = -F.logsigmoid(pos_scores - neg_scores).mean()
        
        reg_loss = ((self.user_emb.weight[users] ** 2).sum(dim=1).mean() +
                    (self.item_emb.weight[pos_items] ** 2).sum(dim=1).mean() +
                    (self.item_emb.weight[neg_items] ** 2).sum(dim=1).mean())
        return bpr, reg_loss
        
    def compute_traitalign_losses(self, user_feat, minilm_vec, scalar_vec, true_proxies):
        pred_proxies = self.mlp(minilm_vec, scalar_vec)
        l_align = F.mse_loss(pred_proxies, true_proxies)
        
        u_emb_norm = F.normalize(user_feat, p=2, dim=1, eps=1e-6)
        sim_emb = torch.matmul(u_emb_norm, u_emb_norm.T)
        true_proxies_norm = F.normalize(true_proxies, p=2, dim=1, eps=1e-6)
        sim_beh = torch.matmul(true_proxies_norm, true_proxies_norm.T)
        
        l_beh_cont = F.mse_loss(sim_emb, sim_beh)
        return l_beh_cont, l_align
        
class BaselineMTLWrapper(nn.Module):
    def __init__(self, num_users, num_items, gnn_type="graphsage", embed_dim=64, minilm_dim=384, scalar_dim=2, out_dim=5):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, embed_dim)
        self.item_emb = nn.Embedding(num_items, embed_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)
        nn.init.xavier_uniform_(self.item_emb.weight)
        
        self.minilm_dim = minilm_dim
        self.W_p = nn.Linear(minilm_dim + scalar_dim, embed_dim)
        self.gnn = ManualHeteroGNN(hidden_channels=embed_dim, gnn_type=gnn_type)
        self.aux_head = nn.Sequential(nn.Linear(embed_dim, out_dim), nn.ReLU())
        
    def forward(self, x_dict, edge_index_dict, minilm_vec, scalar_vec, edge_weight_dict=None):
        u_idx = torch.clamp(x_dict['user'], min=0, max=self.user_emb.num_embeddings - 1)
        i_idx = torch.clamp(x_dict['item'], min=0, max=self.item_emb.num_embeddings - 1)
        
        u_base = self.user_emb(u_idx)
        if self.minilm_dim > 0:
            aux_input = torch.cat([minilm_vec, scalar_vec], dim=1)
        else:
            aux_input = scalar_vec
        u_init = u_base + self.W_p(aux_input)
        
        x_dict_emb = {
            'user': u_init,
            'item': self.item_emb(i_idx)
        }
        return self.gnn(x_dict_emb, edge_index_dict, edge_weight_dict)


## 2. Dataset Loader & Eval Helpers

In [ ]:
def precompute_val_negatives(data, val_edge_index, num_items, num_negatives=99):
    print("Pre-computing 99 fixed negative items for evaluation...")
    val_users = val_edge_index[0].cpu().numpy()
    val_items = val_edge_index[1].cpu().numpy()
    
    train_users = data['user', 'rates', 'item'].edge_index[0].cpu().numpy()
    train_items = data['user', 'rates', 'item'].edge_index[1].cpu().numpy()
    
    train_df = pd.DataFrame({'user': train_users, 'item': train_items})
    train_mask = train_df.groupby('user')['item'].apply(set).to_dict()
    
    user_negatives = {}
    for u, i_true in zip(val_users, val_items):
        if u not in user_negatives:
            negatives = []
            u_train = train_mask.get(u, set())
            while len(negatives) < num_negatives:
                neg = np.random.randint(0, num_items)
                if neg != i_true and neg not in u_train:
                    negatives.append(neg)
            user_negatives[u] = negatives
            
    return user_negatives

def load_traitalign_dataset(dataset_name, data_dir):
    print(f"Loading {dataset_name} dataset from {data_dir}...")
    if dataset_name == 'personality18':
        
        personality_cols = ['openness', 'agreeableness', 'emotional_stability', 'conscientiousness', 'extraversion']
        
        pers_df = pd.read_csv('/kaggle/input/datasets/arslanali4343/top-personality-dataset/2018-personality-data.csv')
        pers_df.columns = pers_df.columns.str.strip()
        ratings_df = pd.read_csv('/kaggle/input/datasets/arslanali4343/top-personality-dataset/2018_ratings.csv')
        ratings_df.columns = ratings_df.columns.str.strip()
        movies_df = pd.read_csv('/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m/movies.csv')
        
        print("Loading MiniLM Embeddings for Items...")
        P_i = np.load('/kaggle/input/datasets/kkaushik06/movielens-minilm-embeddings/P_i.npy')
        movie_emb_dict = {int(row[0]): row[1:] for row in P_i}
        del P_i
        gc.collect()
        
        ratings_df = ratings_df.merge(movies_df[['movieId', 'genres']], left_on='movie_id', right_on='movieId', how='left')
        ratings_df['genres'] = ratings_df['genres'].fillna('Unknown')
        ratings_df['tstamp'] = pd.to_datetime(ratings_df['tstamp'])
        ratings_df = ratings_df.sort_values(['useri', 'tstamp'])
        
        unique_users = ratings_df['useri'].unique()
        unique_movies = ratings_df['movie_id'].unique()
        user_mapping = {id: i for i, id in enumerate(unique_users)}
        movie_mapping = {id: i for i, id in enumerate(unique_movies)}
        
        ratings_df['user_idx'] = ratings_df['useri'].map(user_mapping)
        ratings_df['movie_idx'] = ratings_df['movie_id'].map(movie_mapping)
        pers_df = pers_df[pers_df['userid'].isin(unique_users)].copy()
        pers_df['user_idx'] = pers_df['userid'].map(user_mapping)
        
        num_users = len(user_mapping)
        num_items = len(movie_mapping)
        
        personality_tensor = torch.zeros((num_users, 5), dtype=torch.float)
        for _, row in pers_df.iterrows():
            idx = row['user_idx']
            traits = [row[c] for c in personality_cols]
            personality_tensor[idx] = torch.tensor(traits, dtype=torch.float)
            
        del movies_df
        gc.collect()
        
        print("Constructing Behavioral Profiles...")
        behavioral_profiles = np.zeros((num_users, 5))
        item_counts = ratings_df['movie_idx'].value_counts()
        item_probs = item_counts / item_counts.sum()
        
        grouped = ratings_df.groupby('user_idx')
        for u_idx, group in grouped:
            if len(group) < 2: continue
            probs = group['movie_idx'].map(item_probs).values
            novelty = np.mean(-np.log(probs + 1e-9))
            
            all_genres = []
            for g_str in group['genres']:
                all_genres.extend(g_str.split('|'))
            if len(all_genres) == 0: continue
                
            genre_counts = pd.Series(all_genres).value_counts()
            genre_probs = genre_counts / genre_counts.sum()
            cross_category = entropy(genre_probs)
            
            top_3 = genre_counts.head(3).index.tolist()
            outside_top3 = sum([1 for g in all_genres if g not in top_3])
            exploration = outside_top3 / len(all_genres)
            
            chunk_size = 50
            chunks = [group.iloc[i:i+chunk_size] for i in range(0, len(group), chunk_size)]
            if len(chunks) > 1:
                corrs = []
                for i in range(len(chunks) - 1):
                    c1, c2 = chunks[i], chunks[i+1]
                    genres1 = [g for g_str in c1['genres'] for g in g_str.split('|')]
                    genres2 = [g for g_str in c2['genres'] for g in g_str.split('|')]
                    if genres1 and genres2:
                        vc1 = pd.Series(genres1).value_counts()
                        vc2 = pd.Series(genres2).value_counts()
                        common = set(vc1.index).union(set(vc2.index))
                        if len(common) > 1: 
                            vec1 = [vc1.get(c, 0) for c in common]
                            vec2 = [vc2.get(c, 0) for c in common]
                            corr, _ = spearmanr(vec1, vec2)
                            if not np.isnan(corr):
                                corrs.append(corr)
                temporal_stability = np.mean(corrs) if corrs else 0.0
            else:
                temporal_stability = 0.0
        
            diversity_embs = [movie_emb_dict[m] for m in group['movie_id'] if m in movie_emb_dict]
            if len(diversity_embs) > 1:
                embs_arr = np.array(diversity_embs)
                norms = np.linalg.norm(embs_arr, axis=1, keepdims=True)
                norms[norms == 0] = 1e-9
                norm_embs = embs_arr / norms
                sim_matrix = np.dot(norm_embs, norm_embs.T)
                dist_matrix = 1.0 - sim_matrix
                triu_indices = np.triu_indices(len(embs_arr), k=1)
                diversity = np.mean(dist_matrix[triu_indices])
            else:
                diversity = 0.0
                
            behavioral_profiles[u_idx] = [diversity, novelty, temporal_stability, exploration, cross_category]
        
        behavioral_profiles = minmax_scale(behavioral_profiles)
        
        degrees = ratings_df['user_idx'].value_counts().sort_index().values
        degree_bins = pd.qcut(degrees, q=10, labels=False, duplicates='drop')
        deconfounded_behavior = np.zeros_like(behavioral_profiles)
        
        for u in range(num_users):
            u_bin = degree_bins[u]
            matched_users = np.where(degree_bins == u_bin)[0]
            if len(matched_users) > 0:
                mean_matched = behavioral_profiles[matched_users].mean(axis=0)
                deconfounded_behavior[u] = behavioral_profiles[u] - mean_matched
        
        true_proxies = torch.tensor(deconfounded_behavior, dtype=torch.float32)
        
        print("Constructing PyG HeteroData...")
        shuffled_ratings = ratings_df.sample(frac=1, random_state=42).reset_index(drop=True)
        split_idx = int(len(shuffled_ratings) * 0.90)
        
        train_df = shuffled_ratings.iloc[:split_idx]
        val_df = shuffled_ratings.iloc[split_idx:]
        
        train_edge_index = torch.tensor([train_df['user_idx'].values, train_df['movie_idx'].values], dtype=torch.long)
        val_edge_index = torch.tensor([val_df['user_idx'].values, val_df['movie_idx'].values], dtype=torch.long)
        
        data = HeteroData()
        data['user'].num_nodes = num_users
        data['item'].num_nodes = num_items
        data['user'].n_id = torch.arange(num_users)
        data['item'].n_id = torch.arange(num_items)
        data['user', 'rates', 'item'].edge_index = train_edge_index
        
        # PRECOMPUTE GLOBAL DEGREE NORMALIZATION WEIGHTS
        deg_u = torch.bincount(train_edge_index[0], minlength=num_users).float()
        deg_i = torch.bincount(train_edge_index[1], minlength=num_items).float()
        deg_u_inv_sqrt = deg_u.pow(-0.5)
        deg_u_inv_sqrt.masked_fill_(deg_u_inv_sqrt == float('inf'), 0)
        deg_i_inv_sqrt = deg_i.pow(-0.5)
        deg_i_inv_sqrt.masked_fill_(deg_i_inv_sqrt == float('inf'), 0)
        edge_weight = deg_u_inv_sqrt[train_edge_index[0]] * deg_i_inv_sqrt[train_edge_index[1]]
        data['user', 'rates', 'item'].edge_weight = edge_weight
        
        data = T.ToUndirected()(data)
        
        user_negatives = precompute_val_negatives(data, val_edge_index, num_items)
        
        # PACKAGING FOR TRAITALIGN
        minilm_vec = torch.empty((num_users, 0))
        scalar_vec = personality_tensor # Ocean scores
        
        return {
            'num_users': num_users, 'num_items': num_items, 
            'data': data, 'val_edge_index': val_edge_index,
            'scalar_vec': scalar_vec, 'minilm_vec': minilm_vec, 'true_proxies': true_proxies,
            'user_negatives': user_negatives
        }
    elif dataset_name == 'movielens':
        prefix = 'ml1m'
        emb_file = 'occupation_embeddings.npy'
        idx_col = 'OccupationID'
        item_col = 'MovieID'
    else:
        prefix = 'lastfm'
        emb_file = 'country_embeddings.npy'
        idx_col = 'CountryIdx'
        item_col = 'ArtistMBID'
        
    meta_df = pd.read_csv(f"{data_dir}/{prefix}_user_metadata.csv")
    proxies_df = pd.read_csv(f"{data_dir}/{prefix}_proxies.csv")
    interactions_df = pd.read_csv(f"{data_dir}/{prefix}_interactions.csv")
    
    merged_users = pd.merge(meta_df, proxies_df, on='UserID', how='inner').sort_values('UserID').reset_index(drop=True)
    num_users = len(merged_users)
    
    user_mapping = {u: i for i, u in enumerate(merged_users['UserID'])}
    interactions_df = interactions_df[interactions_df['UserID'].isin(user_mapping)]
    interactions_df['UserID_idx'] = interactions_df['UserID'].map(user_mapping)
    
    unique_items = interactions_df[item_col].unique()
    item_mapping = {item: i for i, item in enumerate(unique_items)}
    interactions_df['Item_idx'] = interactions_df[item_col].map(item_mapping)
    num_items = len(unique_items)
    
    num_edges = len(interactions_df)
    perm = torch.randperm(num_edges)
    train_size = int(num_edges * 0.90)
    
    train_df = interactions_df.iloc[perm[:train_size].numpy()]
    val_df = interactions_df.iloc[perm[train_size:].numpy()]
    
    train_edge_index = torch.tensor(np.array([train_df['UserID_idx'].values, train_df['Item_idx'].values]), dtype=torch.long)
    val_edge_index = torch.tensor(np.array([val_df['UserID_idx'].values, val_df['Item_idx'].values]), dtype=torch.long)
    
    data = HeteroData()
    data['user'].num_nodes = num_users
    data['item'].num_nodes = num_items
    data['user'].n_id = torch.arange(num_users)
    data['item'].n_id = torch.arange(num_items)
    data['user', 'rates', 'item'].edge_index = train_edge_index
    
    # PRECOMPUTE GLOBAL DEGREE NORMALIZATION WEIGHTS (Crucial for GCN/LightGCN mini-batch stability)
    deg_u = torch.bincount(train_edge_index[0], minlength=num_users).float()
    deg_i = torch.bincount(train_edge_index[1], minlength=num_items).float()
    deg_u_inv_sqrt = deg_u.pow(-0.5)
    deg_u_inv_sqrt.masked_fill_(deg_u_inv_sqrt == float('inf'), 0)
    deg_i_inv_sqrt = deg_i.pow(-0.5)
    deg_i_inv_sqrt.masked_fill_(deg_i_inv_sqrt == float('inf'), 0)
    edge_weight = deg_u_inv_sqrt[train_edge_index[0]] * deg_i_inv_sqrt[train_edge_index[1]]
    data['user', 'rates', 'item'].edge_weight = edge_weight
    
    data = T.ToUndirected()(data)
    
    scalar_vec = torch.tensor(merged_users[['Age_Norm', 'Gender_Idx']].values, dtype=torch.float32)
    proxy_cols = [c for c in proxies_df.columns if c not in ['UserID', 'genre_dist']]
    true_proxies = torch.tensor(merged_users[proxy_cols].values, dtype=torch.float32)
    
    true_proxies = (true_proxies - true_proxies.mean(dim=0)) / (true_proxies.std(dim=0) + 1e-6)
    
    unique_embs = np.load(f"{data_dir}/{prefix}_{emb_file}")
    minilm_vec = torch.tensor(unique_embs[merged_users[idx_col].values], dtype=torch.float32)
    
    user_negatives = precompute_val_negatives(data, val_edge_index, num_items)
    
    return {
        'num_users': num_users, 'num_items': num_items, 
        'data': data, 'val_edge_index': val_edge_index,
        'scalar_vec': scalar_vec, 'minilm_vec': minilm_vec, 'true_proxies': true_proxies,
        'user_negatives': user_negatives
    }

def compute_val_ndcg_sampled(user_embs, item_embs, val_edge_index, user_negatives):
    val_users = val_edge_index[0].cpu().numpy()
    val_items = val_edge_index[1].cpu().numpy()
    scores = torch.matmul(user_embs, item_embs.T).cpu().numpy()
    
    ndcg_list = []
    for u, i_true in zip(val_users, val_items):
        u_scores = scores[u]
        candidates = [i_true] + user_negatives[u]
        candidate_scores = u_scores[candidates]
        
        ranked_indices = np.argsort(-candidate_scores)
        rank_of_true = np.where(ranked_indices == 0)[0][0]
        
        if rank_of_true < 10:
            ndcg_list.append(1.0 / np.log2(rank_of_true + 2))
        else:
            ndcg_list.append(0.0)
            
    return np.mean(ndcg_list)

def compute_all_metrics_sampled(user_embs, item_embs, val_edge_index, user_negatives):
    val_users = val_edge_index[0].cpu().numpy()
    val_items = val_edge_index[1].cpu().numpy()
    scores = torch.matmul(user_embs, item_embs.T).cpu().numpy()
    
    ndcg_10_list, ndcg_5_list = [], []
    hr_10_list, hr_5_list = [], []
    
    for u, i_true in zip(val_users, val_items):
        u_scores = scores[u]
        candidates = [i_true] + user_negatives[u]
        candidate_scores = u_scores[candidates]
        
        ranked_indices = np.argsort(-candidate_scores)
        rank_of_true = np.where(ranked_indices == 0)[0][0]
        
        # @10
        if rank_of_true < 10:
            ndcg_10_list.append(1.0 / np.log2(rank_of_true + 2))
            hr_10_list.append(1.0)
        else:
            ndcg_10_list.append(0.0)
            hr_10_list.append(0.0)
            
        # @5
        if rank_of_true < 5:
            ndcg_5_list.append(1.0 / np.log2(rank_of_true + 2))
            hr_5_list.append(1.0)
        else:
            ndcg_5_list.append(0.0)
            hr_5_list.append(0.0)
            
    # HR == Recall because there is only 1 true item per user in validation
    return {
        'NDCG@10': np.mean(ndcg_10_list),
        'NDCG@5': np.mean(ndcg_5_list),
        'HR@10': np.mean(hr_10_list),
        'HR@5': np.mean(hr_5_list),
        'Recall@10': np.mean(hr_10_list),
        'Recall@5': np.mean(hr_5_list)
    }


## 3. Optuna Training Engine & Baseline Implementation

In [ ]:
import copy
BEST_NDCG = -1.0

def run_optuna_trial(trial, dataset_dict, gnn_type, dry_run=True, device="cuda"):
    global BEST_NDCG
    torch.cuda.empty_cache()
    
    lambda_beh_cont = trial.suggest_float("lambda_beh_cont", 1e-4, 1.0, log=True)
    lambda_align    = trial.suggest_float("lambda_align", 1e-4, 1.0, log=True)
    lambda_reg      = trial.suggest_float("lambda_reg", 1e-5, 1e-2, log=True)
    
    data = dataset_dict['data']
    batch_size = 2048
    
    edge_label_index = data["user", "rates", "item"].edge_index
    edge_label = torch.ones(edge_label_index.size(1), dtype=torch.float)
    
    train_loader = LinkNeighborLoader(
        data=data,
        num_neighbors=[20, 10], 
        neg_sampling_ratio=1.0,
        edge_label_index=(("user", "rates", "item"), edge_label_index),
        edge_label=edge_label,
        batch_size=batch_size, 
        shuffle=True,
        num_workers=0
    )
    
    minilm_vec_full = dataset_dict['minilm_vec'].to(device)
    scalar_vec_full = dataset_dict['scalar_vec'].to(device)
    true_proxies_full = dataset_dict['true_proxies'].to(device)
    user_negatives = dataset_dict['user_negatives']
    
    model = TraitAlignWrapper(
        num_users=dataset_dict['num_users'], 
        num_items=dataset_dict['num_items'], 
        gnn_type=gnn_type, 
        embed_dim=64,
        minilm_dim=dataset_dict['minilm_vec'].shape[1],
        scalar_dim=dataset_dict['scalar_vec'].shape[1],
        out_dim=dataset_dict['true_proxies'].shape[1]
    ).to(device)
    
    # REDUCED LEARNING RATE FOR GCN TO PREVENT INSTABILITY
    lr = 0.001 if gnn_type == "gcn" else 0.003
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    
    epochs = 2 if dry_run else 50
    best_val_ndcg = -1.0
    best_model_state = None
    patience_counter = 0
    start_time = time.time()
    
    print(f"\n>>> Starting Optuna Trial {trial.number} (Epochs: {epochs}, Batches: {len(train_loader)}) <<<\n")
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False)
        
        for batch in pbar:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            x_dict_batch = {'user': batch['user'].n_id, 'item': batch['item'].n_id}
            
            safe_edge_index_dict = {}
            edge_weight_dict = {}
            for k, e_idx in batch.edge_index_dict.items():
                src_type, _, dst_type = k
                safe_e_idx = e_idx.clone()
                safe_e_idx[0] = torch.clamp(safe_e_idx[0], min=0, max=max(0, batch[src_type].n_id.size(0) - 1))
                safe_e_idx[1] = torch.clamp(safe_e_idx[1], min=0, max=max(0, batch[dst_type].n_id.size(0) - 1))
                safe_edge_index_dict[k] = safe_e_idx
                if hasattr(batch[k], 'edge_weight'):
                    edge_weight_dict[k] = batch[k].edge_weight
            
            if len(edge_weight_dict) == 0: edge_weight_dict = None
            
            out_dict = model(x_dict_batch, safe_edge_index_dict, edge_weight_dict)
            
            sampled_edge_label_index = batch["user", "rates", "item"].edge_label_index
            sampled_edge_label = batch["user", "rates", "item"].edge_label
            
            pos_edges = sampled_edge_label_index[:, sampled_edge_label == 1.0]
            neg_edges = sampled_edge_label_index[:, sampled_edge_label == 0.0]
            
            num_bpr = min(pos_edges.size(1), neg_edges.size(1))
            if num_bpr == 0: continue
                
            pos_edges = pos_edges[:, :num_bpr]
            neg_edges = neg_edges[:, :num_bpr]
            
            max_u_loc = max(0, out_dict['user'].size(0) - 1)
            max_i_loc = max(0, out_dict['item'].size(0) - 1)
            
            user_idx_local = torch.clamp(pos_edges[0], min=0, max=max_u_loc)
            pos_item_idx_local = torch.clamp(pos_edges[1], min=0, max=max_i_loc)
            neg_item_idx_local = torch.clamp(neg_edges[1], min=0, max=max_i_loc)
            
            user_feat = out_dict['user'][user_idx_local]
            pos_item_feat = out_dict['item'][pos_item_idx_local]
            neg_item_feat = out_dict['item'][neg_item_idx_local]
            
            global_users = torch.clamp(batch['user'].n_id[user_idx_local], min=0, max=dataset_dict['num_users'] - 1)
            global_pos_items = torch.clamp(batch['item'].n_id[pos_item_idx_local], min=0, max=dataset_dict['num_items'] - 1)
            global_neg_items = torch.clamp(batch['item'].n_id[neg_item_idx_local], min=0, max=dataset_dict['num_items'] - 1)
            
            b_minilm = minilm_vec_full[global_users]
            b_scalar = scalar_vec_full[global_users]
            b_proxies = true_proxies_full[global_users]
            
            bpr, reg_loss = model.bpr_loss(user_feat, pos_item_feat, neg_item_feat, global_users, global_pos_items, global_neg_items)
            l_beh, l_align = model.compute_traitalign_losses(user_feat, b_minilm, b_scalar, b_proxies)
            loss = bpr + (lambda_beh_cont * l_beh) + (lambda_align * l_align) + (lambda_reg * reg_loss)
            
            if torch.isnan(loss) or torch.isinf(loss):
                print("WARNING: NaN/Inf loss detected! Skipping batch to protect GPU context.")
                continue
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) 
            optimizer.step()
            epoch_loss += loss.item()
            
            pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
            
        if epoch % 5 == 0 or dry_run or epoch == 1:
            model.eval()
            with torch.no_grad():
                x_dict_all = {'user': data['user'].n_id.to(device), 'item': data['item'].n_id.to(device)}
                safe_edge_index_dict = {k: v.to(device) for k, v in data.edge_index_dict.items()}
                edge_weight_dict = {k: v.to(device) for k, v in data.edge_weight_dict.items()} if hasattr(data, 'edge_weight_dict') else None
                if edge_weight_dict is None:
                    edge_weight_dict = {}
                    for k in data.edge_index_dict.keys():
                        if hasattr(data[k], 'edge_weight'):
                            edge_weight_dict[k] = data[k].edge_weight.to(device)
                if len(edge_weight_dict) == 0: edge_weight_dict = None
                
                out_dict_eval = model(x_dict_all, safe_edge_index_dict, edge_weight_dict)
            
            current_val_ndcg = compute_val_ndcg_sampled(
                out_dict_eval['user'], out_dict_eval['item'], 
                dataset_dict['val_edge_index'], user_negatives
            )
            print(f"  Epoch [{epoch}/{epochs}] | Loss: {epoch_loss:.4f} | Val NDCG@10: {current_val_ndcg:.4f}")
            
            if current_val_ndcg > best_val_ndcg:
                best_val_ndcg = current_val_ndcg
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                
            if patience_counter >= 3:
                print(f"  [Early Stopping] Patience reached.")
                break
            
    print(f"Trial Finished | Time: {time.time() - start_time:.1f}s | Best Val NDCG@10: {best_val_ndcg:.4f}")
    
    if best_val_ndcg > BEST_NDCG:
        BEST_NDCG = best_val_ndcg
        torch.save(model.state_dict(), f"best_{gnn_type}_model.pth")
        
    return best_val_ndcg

def run_baseline_model(dataset_dict, gnn_type, dry_run=True, device="cuda"):
    print(f"\n==========================================")
    print(f"🚀 RUNNING PAPER BASELINE ({gnn_type.upper()} + SYNTHETIC NOISE) 🚀")
    print(f"==========================================")
    
    torch.cuda.empty_cache()
    data = dataset_dict['data']
    batch_size = 2048
    
    edge_label_index = data["user", "rates", "item"].edge_index
    edge_label = torch.ones(edge_label_index.size(1), dtype=torch.float)
    
    train_loader = LinkNeighborLoader(
        data=data,
        num_neighbors=[20, 10], 
        neg_sampling_ratio=1.0,
        edge_label_index=(("user", "rates", "item"), edge_label_index),
        edge_label=edge_label,
        batch_size=batch_size, 
        shuffle=True,
        num_workers=0
    )
    
    # --- NOISE INJECTION FOR LINEAR+NOISE BASELINE ---
    raw_minilm_vec = dataset_dict['minilm_vec'].to(device)
    if raw_minilm_vec.shape[1] > 0:
        noise_minilm = torch.randn_like(raw_minilm_vec)
        minilm_vec_full = F.normalize(noise_minilm, p=2, dim=1)
    else:
        minilm_vec_full = raw_minilm_vec
    
    raw_scalar_vec = dataset_dict['scalar_vec'].to(device)
    scalar_mean = raw_scalar_vec.mean(dim=0)
    scalar_std = raw_scalar_vec.std(dim=0)
    scalar_vec_full = torch.randn_like(raw_scalar_vec) * scalar_std + scalar_mean
    
    true_proxies_full = dataset_dict['true_proxies'].to(device)
    user_negatives = dataset_dict['user_negatives']
    
    model = BaselineMTLWrapper(
        num_users=dataset_dict['num_users'], 
        num_items=dataset_dict['num_items'], 
        gnn_type=gnn_type, 
        embed_dim=64,
        minilm_dim=dataset_dict['minilm_vec'].shape[1],
        scalar_dim=dataset_dict['scalar_vec'].shape[1],
        out_dim=dataset_dict['true_proxies'].shape[1]
    ).to(device)
    
    lr = 0.001 if gnn_type == "gcn" else 0.003
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    lambda_aux = 0.0 # Switching to Linear Integration Strategy (Paper Table 4)
    
    epochs = 2 if dry_run else 50
    best_val_ndcg = -1.0
    best_model_state = None
    patience_counter = 0
    start_time = time.time()
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Baseline Epoch {epoch}/{epochs}", leave=False)
        
        for batch in pbar:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            x_dict_batch = {'user': batch['user'].n_id, 'item': batch['item'].n_id}
            
            safe_edge_index_dict = {}
            edge_weight_dict = {}
            for k, e_idx in batch.edge_index_dict.items():
                src_type, _, dst_type = k
                safe_e_idx = e_idx.clone()
                safe_e_idx[0] = torch.clamp(safe_e_idx[0], min=0, max=max(0, batch[src_type].n_id.size(0) - 1))
                safe_e_idx[1] = torch.clamp(safe_e_idx[1], min=0, max=max(0, batch[dst_type].n_id.size(0) - 1))
                safe_edge_index_dict[k] = safe_e_idx
                if hasattr(batch[k], 'edge_weight'):
                    edge_weight_dict[k] = batch[k].edge_weight
            
            if len(edge_weight_dict) == 0: edge_weight_dict = None
            
            global_users_batch = torch.clamp(batch['user'].n_id, min=0, max=dataset_dict['num_users'] - 1)
            b_minilm = minilm_vec_full[global_users_batch]
            b_scalar = scalar_vec_full[global_users_batch]
            
            out_dict = model(x_dict_batch, safe_edge_index_dict, b_minilm, b_scalar, edge_weight_dict)
            
            sampled_edge_label_index = batch["user", "rates", "item"].edge_label_index
            sampled_edge_label = batch["user", "rates", "item"].edge_label
            
            pos_edges = sampled_edge_label_index[:, sampled_edge_label == 1.0]
            neg_edges = sampled_edge_label_index[:, sampled_edge_label == 0.0]
            
            num_bpr = min(pos_edges.size(1), neg_edges.size(1))
            if num_bpr == 0: continue
                
            pos_edges = pos_edges[:, :num_bpr]
            neg_edges = neg_edges[:, :num_bpr]
            
            max_u_loc = max(0, out_dict['user'].size(0) - 1)
            max_i_loc = max(0, out_dict['item'].size(0) - 1)
            
            user_idx_local = torch.clamp(pos_edges[0], min=0, max=max_u_loc)
            pos_item_idx_local = torch.clamp(pos_edges[1], min=0, max=max_i_loc)
            neg_item_idx_local = torch.clamp(neg_edges[1], min=0, max=max_i_loc)
            
            user_feat = out_dict['user'][user_idx_local]
            pos_item_feat = out_dict['item'][pos_item_idx_local]
            neg_item_feat = out_dict['item'][neg_item_idx_local]
            
            # 1. BPR Loss
            pos_scores = (user_feat * pos_item_feat).sum(dim=1)
            neg_scores = (user_feat * neg_item_feat).sum(dim=1)
            loss_link = -F.logsigmoid(pos_scores - neg_scores).mean()
            
            # 2. MTL Auxiliary Loss (Predict true proxies from final embedding)
            p_hat_u = model.aux_head(out_dict['user'])
            p_u_true = true_proxies_full[global_users_batch]
            
            variance = p_u_true.var()
            if variance > 0:
                loss_aux = F.mse_loss(p_hat_u, p_u_true) / variance
            else:
                loss_aux = F.mse_loss(p_hat_u, p_u_true)
                
            loss = loss_link + (lambda_aux * loss_aux)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) 
            optimizer.step()
            epoch_loss += loss.item()
            
            pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
            
        if epoch % 5 == 0 or dry_run or epoch == 1:
            model.eval()
            with torch.no_grad():
                x_dict_all = {'user': data['user'].n_id.to(device), 'item': data['item'].n_id.to(device)}
                safe_edge_index_dict = {k: v.to(device) for k, v in data.edge_index_dict.items()}
                edge_weight_dict = {}
                for k in data.edge_index_dict.keys():
                    if hasattr(data[k], 'edge_weight'):
                        edge_weight_dict[k] = data[k].edge_weight.to(device)
                if len(edge_weight_dict) == 0: edge_weight_dict = None
                
                out_dict_eval = model(x_dict_all, safe_edge_index_dict, minilm_vec_full, scalar_vec_full, edge_weight_dict)
                
            current_val_ndcg = compute_val_ndcg_sampled(
                out_dict_eval['user'], out_dict_eval['item'], 
                dataset_dict['val_edge_index'], user_negatives
            )
            print(f"  Epoch [{epoch}/{epochs}] | Loss: {epoch_loss:.4f} | Val NDCG@10: {current_val_ndcg:.4f}")
            
            if current_val_ndcg > best_val_ndcg:
                best_val_ndcg = current_val_ndcg
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
                
            if patience_counter >= 3:
                print(f"  [Early Stopping] Patience reached.")
                break
                
    print(f"Baseline Finished | Time: {time.time() - start_time:.1f}s | Best Val NDCG@10: {best_val_ndcg:.4f}")
    model.load_state_dict(best_model_state)
    model.eval()
    with torch.no_grad():
        x_dict_all = {'user': dataset_dict['data']['user'].n_id.to(device), 'item': dataset_dict['data']['item'].n_id.to(device)}
        safe_edge_index_dict = {k: v.to(device) for k, v in dataset_dict['data'].edge_index_dict.items()}
        edge_weight_dict = {}
        for k in dataset_dict['data'].edge_index_dict.keys():
            if hasattr(dataset_dict['data'][k], 'edge_weight'):
                edge_weight_dict[k] = dataset_dict['data'][k].edge_weight.to(device)
        if len(edge_weight_dict) == 0: edge_weight_dict = None
        out_dict_eval = model(x_dict_all, safe_edge_index_dict, minilm_vec_full, scalar_vec_full, edge_weight_dict)
    return compute_all_metrics_sampled(out_dict_eval['user'], out_dict_eval['item'], dataset_dict['val_edge_index'], user_negatives)

def run_traitalign_noise_experiment(dataset_dict, gnn_type, best_params, dry_run=True, device="cuda"):
    print(f"\n==========================================")
    print(f"🚀 RUNNING TRAITALIGN + NOISE ABLATION 🚀")
    print(f"==========================================")
    
    torch.cuda.empty_cache()
    
    lambda_beh_cont = best_params['lambda_beh_cont']
    lambda_align = best_params['lambda_align']
    lambda_reg = best_params['lambda_reg']
    
    data = dataset_dict['data']
    batch_size = 2048
    
    edge_label_index = data["user", "rates", "item"].edge_index
    edge_label = torch.ones(edge_label_index.size(1), dtype=torch.float)
    
    train_loader = LinkNeighborLoader(
        data=data,
        num_neighbors=[20, 10], 
        neg_sampling_ratio=1.0,
        edge_label_index=(("user", "rates", "item"), edge_label_index),
        edge_label=edge_label,
        batch_size=batch_size, 
        shuffle=True,
        num_workers=0
    )
    
    # Generate Noise
    raw_minilm_vec = dataset_dict['minilm_vec'].to(device)
    if raw_minilm_vec.shape[1] > 0:
        noise_minilm = torch.randn_like(raw_minilm_vec)
        minilm_vec_full = F.normalize(noise_minilm, p=2, dim=1)
    else:
        minilm_vec_full = raw_minilm_vec
    
    raw_scalar_vec = dataset_dict['scalar_vec'].to(device)
    scalar_mean = raw_scalar_vec.mean(dim=0)
    scalar_std = raw_scalar_vec.std(dim=0)
    scalar_vec_full = torch.randn_like(raw_scalar_vec) * scalar_std + scalar_mean
    
    true_proxies_full = dataset_dict['true_proxies'].to(device)
    user_negatives = dataset_dict['user_negatives']
    
    model = TraitAlignWrapper(
        num_users=dataset_dict['num_users'], 
        num_items=dataset_dict['num_items'], 
        gnn_type=gnn_type, 
        embed_dim=64,
        minilm_dim=dataset_dict['minilm_vec'].shape[1],
        scalar_dim=dataset_dict['scalar_vec'].shape[1],
        out_dim=dataset_dict['true_proxies'].shape[1]
    ).to(device)
    
    lr = 0.001 if gnn_type == "gcn" else 0.003
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    
    epochs = 2 if dry_run else 50
    best_val_ndcg = -1.0
    best_model_state = None
    patience_counter = 0
    start_time = time.time()
    
    for epoch in range(1, epochs + 1):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(train_loader, desc=f"TraitAlign+Noise Epoch {epoch}/{epochs}", leave=False)
        for batch in pbar:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            x_dict_batch = {'user': batch['user'].n_id, 'item': batch['item'].n_id}
            safe_edge_index_dict = {}
            edge_weight_dict = {}
            for k, e_idx in batch.edge_index_dict.items():
                src_type, _, dst_type = k
                safe_e_idx = e_idx.clone()
                safe_e_idx[0] = torch.clamp(safe_e_idx[0], min=0, max=max(0, batch[src_type].n_id.size(0) - 1))
                safe_e_idx[1] = torch.clamp(safe_e_idx[1], min=0, max=max(0, batch[dst_type].n_id.size(0) - 1))
                safe_edge_index_dict[k] = safe_e_idx
                if hasattr(batch[k], 'edge_weight'):
                    edge_weight_dict[k] = batch[k].edge_weight
            
            if len(edge_weight_dict) == 0: edge_weight_dict = None
            
            out_dict = model(x_dict_batch, safe_edge_index_dict, edge_weight_dict)
            
            sampled_edge_label_index = batch["user", "rates", "item"].edge_label_index
            sampled_edge_label = batch["user", "rates", "item"].edge_label
            
            pos_edges = sampled_edge_label_index[:, sampled_edge_label == 1.0]
            neg_edges = sampled_edge_label_index[:, sampled_edge_label == 0.0]
            
            num_bpr = min(pos_edges.size(1), neg_edges.size(1))
            if num_bpr == 0: continue
            pos_edges = pos_edges[:, :num_bpr]
            neg_edges = neg_edges[:, :num_bpr]
            
            max_u_loc = max(0, out_dict['user'].size(0) - 1)
            max_i_loc = max(0, out_dict['item'].size(0) - 1)
            
            user_idx_local = torch.clamp(pos_edges[0], min=0, max=max_u_loc)
            pos_item_idx_local = torch.clamp(pos_edges[1], min=0, max=max_i_loc)
            neg_item_idx_local = torch.clamp(neg_edges[1], min=0, max=max_i_loc)
            
            user_feat = out_dict['user'][user_idx_local]
            pos_item_feat = out_dict['item'][pos_item_idx_local]
            neg_item_feat = out_dict['item'][neg_item_idx_local]
            
            global_users_batch = torch.clamp(batch['user'].n_id[user_idx_local], min=0, max=dataset_dict['num_users'] - 1)
            b_minilm = minilm_vec_full[global_users_batch]
            b_scalar = scalar_vec_full[global_users_batch]
            
            bpr, loss_reg = model.bpr_loss(
                user_feat, pos_item_feat, neg_item_feat,
                user_idx_local, pos_item_idx_local, neg_item_idx_local
            )
            
            l_beh_cont, l_align = model.compute_traitalign_losses(
                user_feat, b_minilm, b_scalar, true_proxies_full[global_users_batch]
            )
            
            loss = bpr + (lambda_beh_cont * l_beh_cont) + (lambda_align * l_align) + (lambda_reg * loss_reg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()
            epoch_loss += loss.item()
            pbar.set_postfix({'Loss': f"{loss.item():.4f}"})
            
        if epoch % 5 == 0 or dry_run or epoch == 1:
            model.eval()
            with torch.no_grad():
                x_dict_all = {'user': data['user'].n_id.to(device), 'item': data['item'].n_id.to(device)}
                safe_edge_index_dict = {k: v.to(device) for k, v in data.edge_index_dict.items()}
                edge_weight_dict = {}
                for k in data.edge_index_dict.keys():
                    if hasattr(data[k], 'edge_weight'):
                        edge_weight_dict[k] = data[k].edge_weight.to(device)
                if len(edge_weight_dict) == 0: edge_weight_dict = None
                out_dict_eval = model(x_dict_all, safe_edge_index_dict, edge_weight_dict)
                
            current_val_ndcg = compute_val_ndcg_sampled(
                out_dict_eval['user'], out_dict_eval['item'], 
                dataset_dict['val_edge_index'], user_negatives
            )
            print(f"  Epoch [{epoch}/{epochs}] | Loss: {epoch_loss:.4f} | Val NDCG@10: {current_val_ndcg:.4f}")
            
            if current_val_ndcg > best_val_ndcg:
                best_val_ndcg = current_val_ndcg
                best_model_state = copy.deepcopy(model.state_dict())
                patience_counter = 0
            else:
                patience_counter += 1
            if patience_counter >= 3:
                print(f"  [Early Stopping] Patience reached.")
                break
                
    print(f"TraitAlign+Noise Finished | Time: {time.time() - start_time:.1f}s | Best Val NDCG@10: {best_val_ndcg:.4f}")
    model.load_state_dict(best_model_state)
    model.eval()
    with torch.no_grad():
        x_dict_all = {'user': dataset_dict['data']['user'].n_id.to(device), 'item': dataset_dict['data']['item'].n_id.to(device)}
        safe_edge_index_dict = {k: v.to(device) for k, v in dataset_dict['data'].edge_index_dict.items()}
        edge_weight_dict = {}
        for k in dataset_dict['data'].edge_index_dict.keys():
            if hasattr(dataset_dict['data'][k], 'edge_weight'):
                edge_weight_dict[k] = dataset_dict['data'][k].edge_weight.to(device)
        if len(edge_weight_dict) == 0: edge_weight_dict = None
        out_dict_eval = model(x_dict_all, safe_edge_index_dict, edge_weight_dict)
    return compute_all_metrics_sampled(out_dict_eval['user'], out_dict_eval['item'], dataset_dict['val_edge_index'], user_negatives)


## 4. Run Experiment Configuration

In [ ]:

# ==========================================
# CONFIGURATION
# ==========================================
DATASET_NAME = 'personality18'
DATA_DIR = '/kaggle/input/datasets/arslanali4343/top-personality-dataset'
GNN_TYPE = 'graphsage'
DRY_RUN = True
# ==========================================

print(f"🚀 STARTING EXPERIMENT 🚀")
real_dataset = load_traitalign_dataset(DATASET_NAME, DATA_DIR)
print("✅ Real data loaded successfully!")

n_trials = 2 if DRY_RUN else 20
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: run_optuna_trial(trial, real_dataset, GNN_TYPE, dry_run=DRY_RUN), n_trials=n_trials)

print(f"\n🎉 TRAITALIGN TUNING COMPLETE 🎉")
print(f"Best TraitAlign Validation NDCG@10: {study.best_value:.4f}")
print(f"Best TraitAlign model saved to: best_{GNN_TYPE}_model.pth")

# 1. Evaluate Best TraitAlign Model on All Metrics
device = "cuda" if torch.cuda.is_available() else "cpu"
best_model = TraitAlignWrapper(
    num_users=real_dataset['num_users'], 
    num_items=real_dataset['num_items'], 
    gnn_type=GNN_TYPE, 
    embed_dim=64,
    minilm_dim=real_dataset['minilm_vec'].shape[1],
    scalar_dim=real_dataset['scalar_vec'].shape[1],
    out_dim=real_dataset['true_proxies'].shape[1]
).to(device)
best_model.load_state_dict(torch.load(f"best_{GNN_TYPE}_model.pth"))
best_model.eval()
with torch.no_grad():
    x_dict_all = {'user': real_dataset['data']['user'].n_id.to(device), 'item': real_dataset['data']['item'].n_id.to(device)}
    safe_edge_index_dict = {k: v.to(device) for k, v in real_dataset['data'].edge_index_dict.items()}
    edge_weight_dict = {}
    for k in real_dataset['data'].edge_index_dict.keys():
        if hasattr(real_dataset['data'][k], 'edge_weight'):
            edge_weight_dict[k] = real_dataset['data'][k].edge_weight.to(device)
    if len(edge_weight_dict) == 0: edge_weight_dict = None
    out_dict_eval = best_model(x_dict_all, safe_edge_index_dict, edge_weight_dict)
traitalign_metrics = compute_all_metrics_sampled(out_dict_eval['user'], out_dict_eval['item'], real_dataset['val_edge_index'], real_dataset['user_negatives'])

# 2. Train and Evaluate Baseline Model
baseline_metrics = run_baseline_model(real_dataset, GNN_TYPE, dry_run=DRY_RUN)

# 3. Train and Evaluate TraitAlign+Noise
traitalign_noise_metrics = run_traitalign_noise_experiment(real_dataset, GNN_TYPE, study.best_params, dry_run=DRY_RUN)

# 4. Print Summary
print(f"\n🔥 EXPERIMENT FULL SUMMARY 🔥")
print(f"===========================================================")
print(f"{'Model':<20} | {'NDCG@10':<7} | {'NDCG@5':<7} | {'HR@10':<7} | {'HR@5':<7}")
print(f"-----------------------------------------------------------")

def fmt_met(m):
    return f"{m['NDCG@10']:.4f}  | {m['NDCG@5']:.4f}  | {m['HR@10']:.4f}  | {m['HR@5']:.4f}"

print(f"{'Baseline ('+GNN_TYPE+')':<20} | " + fmt_met(baseline_metrics))
print(f"{'TraitAlign ('+GNN_TYPE+')':<20} | " + fmt_met(traitalign_metrics))
print(f"{'TraitAlign+Noise':<20} | " + fmt_met(traitalign_noise_metrics))
print(f"===========================================================")
